# 06 — Baseline survival: demonstracja analizy przeżywalności

Notebook **demonstracyjny** pokazujący, że survival dataset wyprodukowany przez pipeline LUAD-HUBA jest gotowy do analizy przeżywalności. To nie jest pełna analiza naukowa (brak feature selection, cross-validation, optymalizacji modeli) — to dowód użyteczności: na danych z pipeline'u standardowe metody survival (Kaplan-Meier, Cox) po prostu działają.

**Cel pipeline'u był:** przygotować dane TCGA-LUAD do analizy przeżywalności. Ten notebook zamyka tę historię, pokazując że cel został osiągnięty.

**Zakres:**

1. Wczytanie survival_dataset z pipeline'u
2. Przygotowanie (collapse stage, encoding kowariantów)
3. Kaplan-Meier overall (mediana OS, przeżycie 1/3/5 lat)
4. Kaplan-Meier per stage + log-rank
5. Kaplan-Meier dla markera ekspresyjnego (NKX2-1 high vs low)
6. Cox proportional hazards (baseline kliniczny: age + gender + stage)
7. Wnioski

**Decyzje metodologiczne (z EDA + ustalenia projektowe):**
- Endpoint: OS (overall survival), TSS pominięty jako kowariant (EDA: brak batch effectu)
- Próg high/low ekspresji: mediana (najprostszy, brak data-snooping)
- Stage collapse: I/II/III/IV (4 grupy) - Stage IV n=25 za mało na 7-poziomowy podział
- Próbki: tumor_only (już w pipeline), 523 próbki

**Pakiety:** lifelines (już w deps). Jeśli brak: `uv add lifelines`.


## 1. Wczytanie survival dataset

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# Paleta projektu - beże, brązy, zielenie (klimat lab/medical)
PALETTE = {
    "primary": "#5a8b3c",    # zieleń
    "secondary": "#8b5a3c",  # brąz
    "accent": "#c4a484",     # beż
    "dark": "#2a4a1c",       # ciemna zieleń
}

print(f"Projekt: {PROJECT_ROOT}")
print(f"lifelines załadowany | Polars: {pl.__version__}")
print(f"Czas: {datetime.now(timezone.utc).isoformat()}")


In [ ]:
print("=== Wczytuję survival_dataset.parquet ===")
dataset_path = PROCESSED_DIR / "survival_dataset.parquet"
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Brak {dataset_path}\n"
        f"Uruchom najpierw pipeline: luad-huba build-survival --config configs/default.yaml"
    )

ds = pl.read_parquet(dataset_path)
meta_cols = ["sample_id", "case_id", "time", "event", "age_at_index",
             "gender", "ajcc_pathologic_stage", "tissue_type"]
gene_cols = [c for c in ds.columns if c.startswith("ENSG")]

print(f"Dataset: {ds.height} próbek × {ds.width} kolumn")
print(f"  Metadane: {len([c for c in meta_cols if c in ds.columns])} kolumn")
print(f"  Geny: {len(gene_cols)} kolumn")
print()
print("Podstawowe staty przeżycia:")
n_events = ds.filter(pl.col("event")).height
n_censored = ds.filter(~pl.col("event")).height
print(f"  Zdarzenia (zgony): {n_events} ({n_events/ds.height*100:.1f}%)")
print(f"  Cenzurowane:       {n_censored} ({n_censored/ds.height*100:.1f}%)")
print(f"  Czas obserwacji:   {ds['time'].min()}-{ds['time'].max()} dni "
      f"(mediana {ds['time'].median():.0f})")


## 2. Przygotowanie danych

Trzy operacje:
- **Collapse stage** do 4 grup (I/II/III/IV) z podtypów (IA, IB → I itd.). Stage NOS i Unknown jako osobne kategorie (zgodnie z wcześniejszą decyzją - nie wycinamy, oznaczamy).
- **Konwersja do pandas** (lifelines pracuje na pandas, nie polars).
- **Encoding** gender i stage dla Cox.


In [ ]:
def collapse_stage(stage: str) -> str:
    """Collapse szczegółowych stadiów do 4 głównych grup + NOS/Unknown."""
    if stage is None:
        return "Unknown"
    s = stage.replace("Stage ", "")
    if s in ("IV", "IVA", "IVB"):
        return "IV"
    if s.startswith("III"):
        return "III"
    if s.startswith("II"):
        return "II"
    if s.startswith("I"):
        return "I"
    return "Unknown"


# Surowy stage przed collapse - sprawdzenie
print("=== Stage przed collapse ===")
print(ds.group_by("ajcc_pathologic_stage").len().sort("ajcc_pathologic_stage"))

ds = ds.with_columns(
    pl.col("ajcc_pathologic_stage")
    .map_elements(collapse_stage, return_dtype=pl.Utf8)
    .alias("stage_group")
)

print()
print("=== Stage po collapse (4 grupy + NOS/Unknown) ===")
print(ds.group_by("stage_group").len().sort("stage_group"))


In [ ]:
# Konwersja do pandas dla lifelines
# Bierzemy tylko kolumny potrzebne do analizy (metadane + wybrane geny później)
pdf = ds.select(meta_cols + ["stage_group"]).to_pandas()

# time w latach (z dni) dla czytelności wykresów
pdf["time_years"] = pdf["time"] / 365.25

print(f"Pandas DataFrame: {pdf.shape}")
print()
print("Typy danych:")
print(pdf[["time", "time_years", "event", "age_at_index", "gender", "stage_group"]].dtypes)
print()
print("Podgląd:")
print(pdf[["case_id", "time_years", "event", "age_at_index", "gender", "stage_group"]].head())


## 3. Kaplan-Meier — cała kohorta

Podstawowa krzywa przeżycia dla wszystkich pacjentów. Mediana OS i przeżycie w punktach 1, 3, 5 lat.


In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(pdf["time_years"], pdf["event"], label="Cała kohorta")

fig, ax = plt.subplots(figsize=(9, 6))
kmf.plot_survival_function(ax=ax, color=PALETTE["primary"], linewidth=2.5, ci_alpha=0.15)
ax.set_xlabel("Czas (lata)")
ax.set_ylabel("Prawdopodobieństwo przeżycia")
ax.set_title("Kaplan-Meier — cała kohorta TCGA-LUAD")
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3)

# Mediana OS
median_os = kmf.median_survival_time_
ax.axhline(0.5, color=PALETTE["secondary"], linestyle="--", alpha=0.5)
if not np.isnan(median_os):
    ax.axvline(median_os, color=PALETTE["secondary"], linestyle="--", alpha=0.5)
    ax.annotate(f"Mediana OS: {median_os:.2f} lat",
                xy=(median_os, 0.5), xytext=(median_os + 0.5, 0.6),
                fontsize=10, color=PALETTE["dark"])

plt.tight_layout()
plt.show()

print(f"=== Statystyki przeżycia ===")
print(f"Mediana OS: {median_os:.2f} lat" if not np.isnan(median_os) else "Mediana OS: nieosiągnięta")
print()
for years in [1, 3, 5]:
    surv = kmf.survival_function_at_times(years).iloc[0]
    print(f"  Przeżycie {years}-letnie: {surv*100:.1f}%")


## 4. Kaplan-Meier per stage

Czy stadium różnicuje przeżycie? Powinno — to podstawowy sanity check kliniczny (zaawansowane stadia = gorsze rokowanie). Log-rank test sprawdza istotność różnic między grupami.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))

stage_order = ["I", "II", "III", "IV"]
stage_colors = ["#5a8b3c", "#a0843c", "#8b5a3c", "#7a2a1a"]  # zieleń -> brąz -> czerwień

for stage, color in zip(stage_order, stage_colors):
    mask = pdf["stage_group"] == stage
    n = mask.sum()
    if n < 5:
        continue
    kmf_s = KaplanMeierFitter()
    kmf_s.fit(pdf.loc[mask, "time_years"], pdf.loc[mask, "event"],
              label=f"Stage {stage} (n={n})")
    kmf_s.plot_survival_function(ax=ax, color=color, linewidth=2, ci_show=False)

ax.set_xlabel("Czas (lata)")
ax.set_ylabel("Prawdopodobieństwo przeżycia")
ax.set_title("Kaplan-Meier per stage (grupy główne)")
ax.set_ylim(0, 1.0)
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Log-rank test - czy różnice między stage są istotne
main_stages = pdf[pdf["stage_group"].isin(stage_order)].copy()
results = multivariate_logrank_test(
    main_stages["time_years"],
    main_stages["stage_group"],
    main_stages["event"],
)
print("=== Log-rank test (różnice między stage I/II/III/IV) ===")
print(f"  Test statistic: {results.test_statistic:.2f}")
print(f"  p-value: {results.p_value:.2e}")
print()
if results.p_value < 0.05:
    print("  Stadium istotnie różnicuje przeżycie (p < 0.05) - zgodne z oczekiwaniem klinicznym.")
else:
    print("  Brak istotnej różnicy (p >= 0.05) - nietypowe, wymaga sprawdzenia.")


## 5. Kaplan-Meier dla markera ekspresyjnego — NKX2-1

**Najważniejsza sekcja z perspektywy celu pipeline'u** — pokazuje, że pipeline łączy ekspresję genów z danymi przeżycia, i że standardowa analiza (stratyfikacja high/low + log-rank) działa.

NKX2-1 (TTF1) to marker różnicowania pęcherzykowego LUAD. Utrata jego ekspresji wiąże się klinicznie z gorszym rokowaniem. Sprawdzamy, czy widać to w danych z pipeline'u: pacjenci podzieleni na high/low względem **mediany** ekspresji NKX2-1.


In [ ]:
# NKX2-1 ENSG00000136352 - znajdź kolumnę (z sufiksem wersji)
NKX2_1_BASE = "ENSG00000136352"
nkx_col = None
for c in gene_cols:
    if c.startswith(NKX2_1_BASE + ".") or c == NKX2_1_BASE:
        nkx_col = c
        break

if nkx_col is None:
    print(f"NKX2-1 ({NKX2_1_BASE}) nie znaleziony w genach datasetu!")
else:
    print(f"NKX2-1 znaleziony: {nkx_col}")
    # Dodaj ekspresję NKX2-1 do pandas (log2 TPM+1 dla czytelności, ale dataset
    # ma surowe wartości metryki - sprawdzamy jaka metryka)
    nkx_expr = ds[nkx_col].to_numpy()
    pdf["nkx2_1"] = nkx_expr
    pdf["nkx2_1_log"] = np.log2(nkx_expr + 1)

    median_nkx = np.median(nkx_expr)
    pdf["nkx2_1_group"] = np.where(pdf["nkx2_1"] >= median_nkx, "High", "Low")

    print(f"  Mediana ekspresji NKX2-1: {median_nkx:.2f}")
    print(f"  High (>= mediana): {(pdf['nkx2_1_group'] == 'High').sum()} pacjentów")
    print(f"  Low (< mediana):   {(pdf['nkx2_1_group'] == 'Low').sum()} pacjentów")


In [ ]:
if nkx_col is not None:
    fig, ax = plt.subplots(figsize=(9.5, 6.5))

    for group, color in [("High", PALETTE["primary"]), ("Low", PALETTE["secondary"])]:
        mask = pdf["nkx2_1_group"] == group
        n = mask.sum()
        kmf_g = KaplanMeierFitter()
        kmf_g.fit(pdf.loc[mask, "time_years"], pdf.loc[mask, "event"],
                  label=f"NKX2-1 {group} (n={n})")
        kmf_g.plot_survival_function(ax=ax, color=color, linewidth=2.5, ci_alpha=0.12)

    ax.set_xlabel("Czas (lata)")
    ax.set_ylabel("Prawdopodobieństwo przeżycia")
    ax.set_title("Kaplan-Meier — NKX2-1 high vs low (podział: mediana)")
    ax.set_ylim(0, 1.0)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Log-rank high vs low
    high_mask = pdf["nkx2_1_group"] == "High"
    low_mask = pdf["nkx2_1_group"] == "Low"
    lr = logrank_test(
        pdf.loc[high_mask, "time_years"], pdf.loc[low_mask, "time_years"],
        pdf.loc[high_mask, "event"], pdf.loc[low_mask, "event"],
    )
    print(f"=== Log-rank test (NKX2-1 high vs low) ===")
    print(f"  Test statistic: {lr.test_statistic:.2f}")
    print(f"  p-value: {lr.p_value:.4f}")
    print()
    if lr.p_value < 0.05:
        print(f"  Ekspresja NKX2-1 istotnie różnicuje przeżycie (p={lr.p_value:.4f}).")
        print("  Pipeline poprawnie łączy ekspresję z danymi survival.")
    else:
        print(f"  Brak istotnej różnicy (p={lr.p_value:.4f}). To OK dla baseline -")
        print("  pojedynczy gen nie musi być prognostyczny. Ważne że mechanizm działa.")


## 6. Cox proportional hazards — baseline kliniczny

Model Coxa z kowariantami klinicznymi: wiek, płeć, stadium. **TSS pominięty** — EDA wykazała brak batch effectu, więc nie ma potrzeby kontrolować ośrodka. To dowód, że kowarianty z pipeline'u są poprawnie zakodowane i fitują standardowy model.


In [ ]:
# Przygotowanie do Cox - encoding
cox_df = pdf[pdf["stage_group"].isin(stage_order)].copy()  # tylko główne stadia

# Encoding: gender binarnie, stage jako uporządkowane (I=1 ... IV=4)
cox_df["gender_male"] = (cox_df["gender"] == "male").astype(int)
stage_map = {"I": 1, "II": 2, "III": 3, "IV": 4}
cox_df["stage_numeric"] = cox_df["stage_group"].map(stage_map)

# Dataset do Cox: time, event + kowarianty
cox_input = cox_df[["time_years", "event", "age_at_index", "gender_male", "stage_numeric"]].copy()
cox_input = cox_input.dropna()

print(f"=== Cox input: {cox_input.shape[0]} pacjentów ===")
print(f"  (z {pdf.shape[0]} - odrzucono NOS/Unknown stage i braki)")
print()
print("Kowarianty:")
print(f"  age_at_index: {cox_input['age_at_index'].min():.0f}-{cox_input['age_at_index'].max():.0f} "
      f"(mean {cox_input['age_at_index'].mean():.1f})")
print(f"  gender_male: {cox_input['gender_male'].sum()} mężczyzn / "
      f"{(cox_input['gender_male']==0).sum()} kobiet")
print(f"  stage_numeric: rozkład {dict(cox_input['stage_numeric'].value_counts().sort_index())}")


In [ ]:
cph = CoxPHFitter()
cph.fit(cox_input, duration_col="time_years", event_col="event")

print("=== Cox proportional hazards - podsumowanie ===")
cph.print_summary()


In [ ]:
# Forest plot hazard ratios
fig, ax = plt.subplots(figsize=(9, 4.5))

summary = cph.summary
hr = summary["exp(coef)"]
ci_lower = summary["exp(coef) lower 95%"]
ci_upper = summary["exp(coef) upper 95%"]
labels_map = {
    "age_at_index": "Wiek (per rok)",
    "gender_male": "Płeć męska",
    "stage_numeric": "Stadium (per poziom)",
}
y_pos = range(len(summary))

ax.errorbar(hr, y_pos, xerr=[hr - ci_lower, ci_upper - hr],
            fmt="o", color=PALETTE["secondary"], markersize=8, capsize=4, linewidth=1.5)
ax.axvline(1.0, color="grey", linestyle="--", alpha=0.6, label="HR=1 (brak efektu)")
ax.set_yticks(y_pos)
ax.set_yticklabels([labels_map.get(idx, idx) for idx in summary.index])
ax.set_xlabel("Hazard Ratio (95% CI)")
ax.set_title("Cox baseline — hazard ratios kowariantów klinicznych")
ax.legend()
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print(f"=== C-index modelu: {cph.concordance_index_:.4f} ===")
print("(0.5 = losowy, 1.0 = perfekcyjny; dla baseline klinicznego typowo 0.6-0.7)")


## 7. Wnioski

**Co zostało zademonstrowane:**

Pipeline LUAD-HUBA produkuje survival dataset, na którym standardowe metody analizy przeżywalności działają bez żadnych dodatkowych przekształceń:

1. **Kaplan-Meier overall** — krzywa przeżycia całej kohorty, mediana OS, przeżycie 1/3/5-letnie
2. **KM per stage** — stadium różnicuje przeżycie (log-rank), zgodnie z oczekiwaniem klinicznym — sanity check kohorty
3. **KM dla NKX2-1** — pipeline poprawnie łączy ekspresję genów z danymi survival; stratyfikacja high/low + log-rank działa
4. **Cox baseline** — kowarianty kliniczne (wiek, płeć, stadium) poprawnie zakodowane, model się fituje, hazard ratios interpretowalne, C-index obliczony

**Cel pipeline'u osiągnięty:** dane TCGA-LUAD są przygotowane i gotowe do analizy przeżywalności. Survival dataset zawiera ekspresję (~20k protein_coding genów) zintegrowaną z kowariantami klinicznymi i poprawnie zdefiniowanymi time/event.

**Co NIE wchodziło w zakres tego baseline (możliwe rozszerzenia):**
- Feature selection na pełnej macierzy ekspresji (LASSO, elastic net, univariate screening)
- Cross-validation i optymalizacja hiperparametrów
- Multivariate Cox z genami jako predyktorami
- Progression-free survival (PFI) — wymaga TCGA-CDR (zaplanowane jako wersja 2 pipeline'u)
- Modele uczenia maszynowego / deep learning
- Multimodalność (integracja z obrazami histopatologicznymi)

To są kierunki na przyszłość. Obecny pipeline dostarcza fundament: czyste, zintegrowane, gotowe do modelowania dane.
